<a href="https://colab.research.google.com/github/robertbarcik/MCP-tutorial/blob/main/1_MCP_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP Tutorial - Exploring MCP Server Functions

In this notebook, you'll get a hands-on introduction to the **Model Context Protocol (MCP)** by exploring the functions that power an IT support system. We'll import and call server functions directly to understand what data is available and how each server responds - including how errors are designed to help AI models recover gracefully.

**What you'll learn:**

- What MCP is and why it's useful for building AI-powered applications
- How 5 specialized servers work together to form a complete IT support system
- How each server function behaves with both successful queries and error responses
- Why structured error messages are the key to reliable AI tool use

# What is MCP?

**Model Context Protocol (MCP)** is a standardized protocol for connecting AI models with external tools and data sources. Think of it as a universal adapter between an AI model and the services it needs to interact with.

## Why do we need it?

When building AI-powered applications, we often want the model to interact with external systems like databases, APIs, or file systems. Without a standard protocol, every integration requires custom code. MCP solves this by providing:

- **Tool discovery** - the AI model can ask "what tools are available?" and get a structured list with descriptions and parameter schemas
- **Standardized communication** - every tool follows the same request/response format, regardless of what it does internally
- **Server isolation** - each service runs as an independent process, so failures in one server don't affect others

## Our IT Support System

In this tutorial, we built a complete IT support system using 5 specialized MCP servers. Each server handles a different part of the business:

```
                         ┌──────────────────┐
                         │    AI Model      │
                         │   (gpt-5-nano)   │
                         └────────┬─────────┘
                                  │
                         ┌────────┴─────────┐
                         │  MCP Orchestrator│
                         └────────┬─────────┘
              ┌─────────┬────────┼────────┬─────────┐
              ▼         ▼        ▼        ▼         ▼
         ┌────────┐┌────────┐┌────────┐┌────────┐┌────────┐
         │ Ticket ││Customer││Billing ││  KB    ││ Asset  │
         │ Server ││ Server ││ Server ││ Server ││ Server │
         └────────┘└────────┘└────────┘└────────┘└────────┘
```

- **Ticket Server** - search and manage support tickets, view metrics, find similar tickets
- **Customer Server** - look up customer information, check status, get SLA terms
- **Billing Server** - view invoices, check payment status, calculate outstanding balances
- **Knowledge Base Server** - search for articles and solutions to technical problems
- **Asset Server** - track hardware and software assets, check warranty status

## How we'll explore MCP in this notebook

All server functions in this tutorial are written as **regular Python functions** that can be imported and called directly. In this notebook, we'll start by exploring each server this way - importing its functions and calling them to understand the data structures and error handling patterns.

At the end, we'll look at how the full MCP protocol ties everything together with an AI model.

# Setup

Before running this notebook, make sure the server files are accessible and dependencies are installed.

## Upload Server Files

This notebook imports functions directly from the MCP server files. At the end of the notebook, we'll also use `interactive_client.py` to run the full MCP system from a terminal. Depending on your environment:

**Google Colab:** Upload these 7 Python files using the Files panel on the left sidebar:

1. `ticket_server.py` - Support ticket management
2. `customer_server.py` - Customer database and SLA lookup
3. `billing_server.py` - Invoices and payment tracking
4. `kb_server.py` - Knowledge base search
5. `asset_server.py` - Hardware/software asset tracking
6. `mcp_client.py` - MCP orchestrator (coordinates all servers)
7. `interactive_client.py` - Command-line chat interface

**Local Jupyter:** Make sure all files are in the same directory as this notebook (they should be if you cloned the repository).

## Install Dependencies

We only need the MCP SDK and a few supporting packages. The server functions themselves are pure Python with no additional dependencies.

One thing worth knowing before we install anything: MCP itself is language-independent. The protocol is an agreement about the shape of the messages that travel between client and server (JSON-RPC), not code — much like HTTP is an agreement, not a function. The `mcp` package we install here is the official Python SDK, a ready-made implementation of that agreement; SDKs also exist for TypeScript, Java, C#, Kotlin and other languages, and servers written in different languages interoperate because the same JSON goes over the wire. We write Python because this course runs in Python — a choice, not a requirement.


In [1]:
!pip install -q mcp==1.27.0 nest-asyncio==1.6.0 typing-extensions==4.13.2 2>/dev/null

print("✅ Dependencies installed!")

✅ Dependencies installed!


# Exploring the Server Functions

Now that we have everything set up, let's explore each server's functions. We'll import them directly and call them as regular Python functions.

For each server, we'll look at two things:

- **Successful queries** - what the data looks like when the requested information exists
- **Error responses** - what happens when data doesn't exist, and how the error messages are structured to help the AI model recover

Pay attention to the error responses. They include fields like `suggested_actions` and `follow_up_tools` that tell the AI model what to try next. This is what makes MCP-based systems more reliable than simple API calls.

## Ticket Server

The ticket server manages support tickets for IT issues like Windows BSOD errors, Linux permission problems, and macOS crashes. It provides functions for searching tickets, getting details, viewing metrics, and finding similar tickets.

Let's start by searching for critical priority tickets.

In [2]:
# Import ticket server functions
from ticket_server import search_tickets, get_ticket_details, get_ticket_metrics, find_similar_tickets_to, TICKETS

# Example 1: Search for critical priority tickets (SUCCESS)
print("=" * 60)
print("Example 1: Searching for critical tickets")
print("=" * 60)
critical_tickets = search_tickets(priority="critical")
print(f"Found {critical_tickets['total_count']} critical tickets:\n")
for ticket in critical_tickets['tickets']:
    print(f"  {ticket['ticket_id']}: {ticket['subject']}")

Example 1: Searching for critical tickets
Found 3 critical tickets:

  TKT-1002: Linux server disk full - /var/log consuming 95% space
  TKT-1009: Windows 11 BitLocker recovery key prompt on every boot
  TKT-1011: Windows Server 2019 Active Directory replication failing


Now let's look at how the server handles requests for specific tickets. We'll fetch an existing ticket to see the full data structure, and then try to get one that doesn't exist to see the error response with recovery hints.

In [3]:
# Example 2: Get details of an existing ticket (SUCCESS)
print("\n" + "=" * 60)
print("Example 2: Getting details for existing ticket TKT-1001")
print("=" * 60)
ticket = get_ticket_details("TKT-1001")
print(f"Ticket: {ticket['ticket_id']}")
print(f"Subject: {ticket['subject']}")
print(f"Status: {ticket['status']}")
print(f"Priority: {ticket['priority']}")
print(f"Description: {ticket['description'][:100]}...")

# Example 3: Try to get a non-existent ticket (ERROR WITH HINTS)
print("\n" + "=" * 60)
print("Example 3: Attempting to get non-existent ticket TKT-9999")
print("=" * 60)
error_response = get_ticket_details("TKT-9999")
print("This ticket doesn't exist. Here's the error response with LLM hints:\n")
import json
print(json.dumps(error_response, indent=2))
print("\n💡 Notice the 'suggested_actions' and 'follow_up_tools' that guide the LLM!")


Example 2: Getting details for existing ticket TKT-1001
Ticket: TKT-1001
Subject: Windows 11 BSOD - DRIVER_IRQL_NOT_LESS_OR_EQUAL
Status: open
Priority: high
Description: User experiencing frequent blue screens with error DRIVER_IRQL_NOT_LESS_OR_EQUAL. Occurs during vide...

Example 3: Attempting to get non-existent ticket TKT-9999
This ticket doesn't exist. Here's the error response with LLM hints:

{
  "error": "Ticket TKT-9999 not found",
  "reason": "The ticket_id did not match any tickets in the dataset.",
  "suggested_actions": [
    "Call search_tickets with customer_id or priority filters to rediscover the ticket.",
    "Verify the ticket_id format (e.g., TKT-1001)."
  ],
  "retryable": true,
  "follow_up_tools": [
    "search_tickets"
  ],
  "ticket_id": "TKT-9999"
}

💡 Notice the 'suggested_actions' and 'follow_up_tools' that guide the LLM!


The ticket server also provides aggregate metrics that give an overview of the support team's workload over a given time period.

In [4]:
# Example 4: Get ticket metrics (SUCCESS)
print("\n" + "=" * 60)
print("Example 4: Getting ticket metrics for last 7 days")
print("=" * 60)
metrics = get_ticket_metrics("last_7_days")
print(f"Ticket Metrics (Last 7 Days):")
print(f"  Total: {metrics['total_tickets']}")
print(f"  Open: {metrics['open_tickets']}")
print(f"  In Progress: {metrics['in_progress_tickets']}")
print(f"  Resolved: {metrics['resolved_tickets']}")
print(f"  Avg Resolution Time: {metrics['avg_resolution_time_hours']} hours")


Example 4: Getting ticket metrics for last 7 days
Ticket Metrics (Last 7 Days):
  Total: 14
  Open: 6
  In Progress: 5
  Resolved: 3
  Avg Resolution Time: 56.0 hours


## Customer Server

The customer server stores information about the companies we support, including their contact details, account status, and SLA (Service Level Agreement) terms. Each customer has a tier (basic, standard, or premium) that determines their response time guarantees.

Let's look up a customer and see what happens when we search for one that doesn't exist.

In [5]:
# Import customer server functions
from customer_server import lookup_customer, check_customer_status, get_sla_terms, list_customer_contacts

# Example 1: Look up an existing customer (SUCCESS)
print("=" * 60)
print("Example 1: Looking up existing customer CUST-001")
print("=" * 60)
customer = lookup_customer(customer_id="CUST-001")
print(f"Customer: {customer['company_name']}")
print(f"Tier: {customer['tier']}")
print(f"Status: {customer['status']}")
print(f"Account Manager: {customer['account_manager']}")

# Example 2: Try to look up non-existent customer (ERROR WITH HINTS)
print("\n" + "=" * 60)
print("Example 2: Attempting to look up non-existent customer CUST-999")
print("=" * 60)
error_response = lookup_customer(customer_id="CUST-999")
print("This customer doesn't exist. Here's the error response:\n")
import json
print(json.dumps(error_response, indent=2))
print("\n💡 The LLM can use these hints to try a different approach!")

Example 1: Looking up existing customer CUST-001
Customer: TechCorp Industries
Tier: premium
Status: active
Account Manager: Alice Johnson

Example 2: Attempting to look up non-existent customer CUST-999
This customer doesn't exist. Here's the error response:

{
  "error": "Customer not found",
  "reason": "No customer record matched the provided identifiers.",
  "suggested_actions": [
    "Double-check the customer_id, email, or company_name values.",
    "Try using company_name with a partial match (e.g., 'TechCorp')."
  ],
  "retryable": true,
  "follow_up_tools": [
    "lookup_customer"
  ],
  "search_criteria": {
    "customer_id": "CUST-999"
  }
}

💡 The LLM can use these hints to try a different approach!


Let's check the SLA terms for a customer. This information helps the AI model understand response time commitments and prioritize support accordingly.

In [6]:
# Example 3: Get SLA terms for existing customer (SUCCESS)
print("\n" + "=" * 60)
print("Example 3: Getting SLA terms for CUST-001")
print("=" * 60)
sla = get_sla_terms("CUST-001")
print(f"SLA for {sla['company_name']}:")
print(f"  Level: {sla['sla_terms']['level']}")
print(f"  Response Time: {sla['sla_terms']['response_time_hours']} hours")
print(f"  Resolution Time: {sla['sla_terms']['resolution_time_hours']} hours")
print(f"  Support Hours: {sla['sla_terms']['support_hours']}")


Example 3: Getting SLA terms for CUST-001
SLA for TechCorp Industries:
  Level: platinum
  Response Time: 1 hours
  Resolution Time: 8 hours
  Support Hours: 24/7


## Billing Server

The billing server tracks invoices and payments for each customer. It can retrieve invoices by customer or invoice ID, check payment status, and calculate outstanding balances.

Let's get the invoices for a customer and see how the server handles an invalid invoice ID.

In [7]:
# Import billing server functions
from billing_server import get_invoice, check_payment_status, calculate_outstanding_balance

# Example 1: Get invoices for an existing customer (SUCCESS)
print("=" * 60)
print("Example 1: Getting invoices for customer CUST-001")
print("=" * 60)
invoices = get_invoice(customer_id="CUST-001")
print(f"Total invoices for customer: {invoices['total_invoices']}\n")
for inv in invoices['invoices'][:3]:  # Show first 3
    print(f"  {inv['invoice_id']}: ${inv['amount']} - {inv['status']}")

# Example 2: Try to get invoice with invalid ID (ERROR WITH HINTS)
print("\n" + "=" * 60)
print("Example 2: Attempting to get invoice INV-9999 (doesn't exist)")
print("=" * 60)
error_response = get_invoice(invoice_id="INV-9999")
print("This invoice doesn't exist. Here's the error response:\n")
import json
print(json.dumps(error_response, indent=2))
print("\n💡 Notice how the error suggests using customer_id instead!")

Example 1: Getting invoices for customer CUST-001
Total invoices for customer: 2

  INV-2025-001: $450.0 - paid
  INV-2025-004: $600.0 - pending

Example 2: Attempting to get invoice INV-9999 (doesn't exist)
This invoice doesn't exist. Here's the error response:

{
  "error": "Invoice INV-9999 not found",
  "reason": "The provided invoice_id does not exist in the billing dataset.",
  "suggested_actions": [
    "Call calculate_outstanding_balance to review invoices by customer.",
    "Use get_invoice with customer_id to browse available invoices."
  ],
  "retryable": true,
  "follow_up_tools": [
    "calculate_outstanding_balance",
    "get_invoice"
  ],
  "invoice_id": "INV-9999"
}

💡 Notice how the error suggests using customer_id instead!


We can also calculate the total outstanding balance for a customer, which aggregates all unpaid invoices into a single summary.

In [8]:
# Example 3: Calculate outstanding balance (SUCCESS)
print("\n" + "=" * 60)
print("Example 3: Calculating outstanding balance for CUST-002")
print("=" * 60)
balance = calculate_outstanding_balance("CUST-002")
print(f"Outstanding Balance for Customer:")
print(f"  Total: ${balance['outstanding_balance']}")
print(f"  Overdue: ${balance['overdue_amount']}")
print(f"  Unpaid Invoices: {balance['number_of_unpaid_invoices']}")


Example 3: Calculating outstanding balance for CUST-002
Outstanding Balance for Customer:
  Total: $3000.0
  Overdue: $2150.0
  Unpaid Invoices: 3


## Knowledge Base Server

The knowledge base stores technical articles and troubleshooting guides. The AI model uses this server to find solutions for common IT problems. Unlike the other servers, a search with no results returns an empty list rather than an error - the AI model can simply try different search terms.

Let's search for articles about a common issue.

In [9]:
# Import knowledge base server functions
from kb_server import search_solutions, get_article

# Example 1: Search for BSOD articles (SUCCESS)
print("=" * 60)
print("Example 1: Searching for articles about BSOD")
print("=" * 60)
results = search_solutions("BSOD", limit=3)
print(f"Found {results['total_count']} articles about BSOD:\n")
for article in results['results']:
    print(f"  {article['article_id']}: {article['title']}")
    print(f"    Relevance: {article['relevance_score']}, Views: {article['views']}")

# Example 2: Search with no results (EMPTY RESULT - NOT AN ERROR)
print("\n" + "=" * 60)
print("Example 2: Searching for articles about 'xyz123nonexistent'")
print("=" * 60)
no_results = search_solutions("xyz123nonexistent", limit=3)
print(f"Found {no_results['total_count']} articles.")
print("Note: No error - just empty results. LLM can try different search terms.")

Example 1: Searching for articles about BSOD
Found 1 articles about BSOD:

  KB-001: Resolving Windows BSOD DRIVER_IRQL_NOT_LESS_OR_EQUAL
    Relevance: 15, Views: 1523

Example 2: Searching for articles about 'xyz123nonexistent'
Found 0 articles.
Note: No error - just empty results. LLM can try different search terms.


Let's retrieve a full article by its ID and also see what happens when we request one that doesn't exist.

In [10]:
# Example 3: Get an existing article (SUCCESS)
print("\n" + "=" * 60)
print("Example 3: Getting full article KB-001")
print("=" * 60)
article = get_article("KB-001")
print(f"Article: {article['title']}")
print(f"Category: {article['category']}")
print(f"\nContent Preview:")
print(article['content'][:200] + "...")

# Example 4: Try to get non-existent article (ERROR WITH HINTS)
print("\n" + "=" * 60)
print("Example 4: Attempting to get non-existent article KB-999")
print("=" * 60)
error_response = get_article("KB-999")
print("This article doesn't exist. Here's the error response:\n")
import json
print(json.dumps(error_response, indent=2))
print("\n💡 The error suggests using search_solutions to find relevant articles!")


Example 3: Getting full article KB-001
Article: Resolving Windows BSOD DRIVER_IRQL_NOT_LESS_OR_EQUAL
Category: Windows Troubleshooting

Content Preview:

# Resolution Steps

1. Boot into Safe Mode
2. Open Device Manager
3. Update or roll back recently updated drivers
4. Run Windows Memory Diagnostic
5. Check for Windows Updates
6. Use Driver Verifier ...

Example 4: Attempting to get non-existent article KB-999
This article doesn't exist. Here's the error response:

{
  "error": "Article KB-999 not found",
  "reason": "The knowledge base does not include that article_id.",
  "suggested_actions": [
    "Call search_solutions with keywords related to the issue.",
    "Use find_related_articles starting from a known article to explore similar topics."
  ],
  "retryable": true,
  "follow_up_tools": [
    "search_solutions",
    "find_related_articles"
  ],
  "article_id": "KB-999"
}

💡 The error suggests using search_solutions to find relevant articles!


## Asset Server

The asset server tracks hardware and software assets including servers, workstations, and laptops. It also monitors warranty status and software licenses, which is important for planning replacements and renewals.

Let's look up an asset and see what information is available.

In [11]:
# Import asset server functions
from asset_server import lookup_asset, check_warranty

# Example 1: Look up an existing asset (SUCCESS)
print("=" * 60)
print("Example 1: Looking up asset AST-SRV-001")
print("=" * 60)
asset = lookup_asset(asset_id="AST-SRV-001")
print(f"Asset: {asset['hostname']}")
print(f"Type: {asset['asset_type']}")
print(f"Manufacturer: {asset['manufacturer']} {asset['model']}")
print(f"Location: {asset['location']}")

# Example 2: Try to look up non-existent asset (ERROR WITH HINTS)
print("\n" + "=" * 60)
print("Example 2: Attempting to look up non-existent asset AST-999")
print("=" * 60)
error_response = lookup_asset(asset_id="AST-999")
print("This asset doesn't exist. Here's the error response:\n")
import json
print(json.dumps(error_response, indent=2))
print("\n💡 Error provides context and suggests alternative approaches!")

Example 1: Looking up asset AST-SRV-001
Asset: sql-prod-01.dataflow.local
Type: server
Manufacturer: HPE ProLiant DL380 Gen10
Location: DataFlow Data Center - Rack 12

Example 2: Attempting to look up non-existent asset AST-999
This asset doesn't exist. Here's the error response:

{
  "error": "No assets found matching criteria",
  "reason": "The identifiers did not match any assets in the dataset.",
  "suggested_actions": [
    "Provide asset_id or serial_number for an exact match.",
    "Use customer_id alone to list all assets for a customer."
  ],
  "retryable": true,
  "follow_up_tools": [
    "lookup_asset"
  ],
  "search_criteria": {
    "asset_id": "AST-999"
  }
}

💡 Error provides context and suggests alternative approaches!


Finally, let's check the warranty status for an asset. This helps the AI model determine whether a device is still covered and when the warranty expires.

In [12]:
# Example 3: Check warranty for existing asset (SUCCESS)
print("\n" + "=" * 60)
print("Example 3: Checking warranty for asset AST-SRV-001")
print("=" * 60)
warranty = check_warranty("AST-SRV-001")
w = warranty['warranty']
print(f"Warranty for {warranty['hostname']}:")
print(f"  Coverage: {w['coverage_type']}")
print(f"  Status: {w['status']}")
print(f"  End Date: {w['end_date']}")
print(f"  Days Remaining: {w['remaining_days']}")
print(f"  Expired: {w['is_expired']}")


Example 3: Checking warranty for asset AST-SRV-001
Warranty for sql-prod-01.dataflow.local:
  Coverage: 24x7 4-hour response
  Status: active
  End Date: 2027-03-25
  Days Remaining: 257
  Expired: False


# What Happens on the Wire

So far in this notebook we've been calling server functions as plain Python:

```python
from ticket_server import search_tickets
search_tickets(priority="critical")
```

This skips MCP entirely. We're just importing and calling a Python function. In a real MCP system, the client and server communicate through structured JSON messages instead. Those messages are what we call **the wire protocol**.

Understanding the wire format matters for two reasons. It's what makes MCP **portable**. Any client that speaks this format can talk to any MCP server, regardless of what language either side is written in. It's also the fastest way to understand what went wrong when you see unexpected behaviour in a real deployment.

## JSON-RPC 2.0: The Language MCP Speaks

MCP uses **JSON-RPC 2.0** as its wire format. It's a lightweight protocol where every message is a JSON object. There are exactly three message types:

| Type | Has `id`? | Has `method`? | Has `result`? | When it's used |
|------|-----------|---------------|---------------|----------------|
| **Request** | ✅ yes | ✅ yes | ❌ no | Client asks for something; server must reply |
| **Response** | ✅ yes | ❌ no | ✅ yes | Server's answer to a specific request |
| **Notification** | ❌ no | ✅ yes | ❌ no | One-way message; no reply expected |

The `id` field is what keeps requests and responses matched up. When a client sends a request with `"id": 2`, the server puts `"id": 2` in the response. This is true even if other messages crossed the wire in between.

Every MCP interaction follows the same lifecycle, and each step has a specific `method` name:

| Method | Direction | Type | What it does |
|--------|-----------|------|--------------|
| `initialize` | CLIENT → SERVER | Request | Opens the connection, negotiates protocol version and capabilities |
| *(response)* | SERVER → CLIENT | Response | Confirms server capabilities and identity |
| `notifications/initialized` | CLIENT → SERVER | Notification | Signals "I've received your capabilities, ready to proceed" |
| `tools/list` | CLIENT → SERVER | Request | Asks "what tools do you have?" |
| *(response)* | SERVER → CLIENT | Response | Returns all tool definitions with parameter schemas |
| `tools/call` | CLIENT → SERVER | Request | Calls a specific tool with arguments |
| *(response)* | SERVER → CLIENT | Response | Returns the tool's result |

Let's capture these messages in real time.

In [13]:
# Wire protocol demo: setup
#
# We connect a real MCP client to the ticket server using in-memory streams
# instead of a subprocess, so this runs directly in the notebook.
# The LoggingReceiveStream and LoggingSendStream wrappers sit between the
# two sides and record every message that passes through.

import asyncio
import json
import anyio
import nest_asyncio
from mcp import ClientSession
from mcp.shared.memory import create_client_server_memory_streams, SessionMessage
from ticket_server import app as ticket_app

nest_asyncio.apply()

wire_log = []


class LoggingReceiveStream:
    """Wraps a receive stream and records each incoming message."""

    def __init__(self, stream, direction):
        self._stream = stream
        self._direction = direction

    def __getattr__(self, name):
        return getattr(self._stream, name)

    async def receive(self):
        item = await self._stream.receive()
        if isinstance(item, SessionMessage):
            wire_log.append({"dir": self._direction, "msg": item.message})
        return item

    def __aiter__(self): return self

    async def __anext__(self):
        try:
            return await self.receive()
        except (anyio.EndOfStream, StopAsyncIteration):
            raise StopAsyncIteration

    async def aclose(self): await self._stream.aclose()
    async def __aenter__(self): return self
    async def __aexit__(self, *a): await self.aclose()


class LoggingSendStream:
    """Wraps a send stream and records each outgoing message."""

    def __init__(self, stream, direction):
        self._stream = stream
        self._direction = direction

    def __getattr__(self, name):
        return getattr(self._stream, name)

    async def send(self, item):
        if isinstance(item, SessionMessage):
            wire_log.append({"dir": self._direction, "msg": item.message})
        await self._stream.send(item)

    async def aclose(self): await self._stream.aclose()
    async def __aenter__(self): return self
    async def __aexit__(self, *a): await self.aclose()


print("✅ Wire logging infrastructure ready")

✅ Wire logging infrastructure ready


## Running the Demo

The cell below connects an MCP client to the ticket server in three steps. These are the same steps `mcp_client.py` performs for each server every time it starts up:

1. **`session.initialize()`**: handshake: negotiate protocol version, exchange capability lists
2. **`session.list_tools()`**: discovery: ask the server what tools it exposes
3. **`session.call_tool(...)`**: invocation: call `search_tickets` with `priority="critical"`

A note on what we're using here: this demo connects a `ClientSession` directly to the ticket server with in-memory streams. There's no orchestrator and no AI model yet. The goal is to show the raw protocol in isolation. The next section explains what `mcp_client.py` adds on top.

In [14]:
# Helper: trim long messages so the output stays readable
def _smart_format(msg_dict):
    d = dict(msg_dict)
    if "result" in d and isinstance(d["result"], dict):
        result = dict(d["result"])

        # tools/list response: show names only, skip full schemas
        if "tools" in result:
            tools = result["tools"]
            result["tools"] = [
                {"name": t["name"], "description": t["description"][:55] + "..."}
                for t in tools
            ]
            result["_note"] = f"({len(tools)} tools total, inputSchemas trimmed)"
            d["result"] = result

        # tools/call response: show ticket count, not full JSON blob
        if "content" in result and result["content"]:
            text = result["content"][0].get("text", "")
            if len(text) > 120:
                try:
                    parsed = json.loads(text)
                    count = parsed.get("total_count", "?")
                    result["content"] = [{
                        "type": "text",
                        "text": f"{{... {count} tickets returned (trimmed for readability) ...}}"
                    }]
                except Exception:
                    result["content"] = [{"type": "text", "text": text[:120] + "..."}]
            d["result"] = result
    return d


def print_wire_log(log):
    print(f"Captured {len(log)} messages:\n")
    for i, entry in enumerate(log, 1):
        direction = entry["dir"]
        msg       = entry["msg"].model_dump(exclude_none=True)

        has_id     = "id" in msg
        has_method = "method" in msg

        if has_method and has_id:
            msg_type  = "REQUEST"
            type_note = f"(id={msg['id']}, expects a response)"
        elif has_method and not has_id:
            msg_type  = "NOTIFICATION"
            type_note = "(no id, one-way, no response expected)"
        else:
            msg_type  = "RESPONSE"
            type_note = f"(answers request id={msg.get('id')})"

        print(f"{'=' * 66}")
        print(f"  [{i}]  {direction}  |  {msg_type}  {type_note}")
        print(f"{'-' * 66}")
        print(json.dumps(_smart_format(msg), indent=2))
        print()


# Run the demo
async def run_wire_demo():
    wire_log.clear()
    async with create_client_server_memory_streams() as (client_streams, server_streams):
        client_read, client_write = client_streams
        server_read,  server_write  = server_streams

        async with anyio.create_task_group() as tg:
            # Server runs on unlogged streams (we only need the client side)
            tg.start_soon(lambda: ticket_app.run(
                server_read, server_write,
                ticket_app.create_initialization_options()
            ))

            # Client runs on logged streams; every message is captured
            async with ClientSession(
                read_stream=LoggingReceiveStream(client_read,  "SERVER → CLIENT"),
                write_stream=LoggingSendStream(client_write, "CLIENT → SERVER"),
            ) as session:
                await session.initialize()                               # step 1
                await session.list_tools()                               # step 2
                await session.call_tool(                                 # step 3
                    "search_tickets", {"priority": "critical"}
                )
            tg.cancel_scope.cancel()

asyncio.run(run_wire_demo())
print_wire_log(wire_log)

Captured 7 messages:

  [1]  CLIENT → SERVER  |  REQUEST  (id=0, expects a response)
------------------------------------------------------------------
{
  "method": "initialize",
  "params": {
    "protocolVersion": "2025-11-25",
    "capabilities": {},
    "clientInfo": {
      "name": "mcp",
      "version": "0.1.0"
    }
  },
  "jsonrpc": "2.0",
  "id": 0
}

  [2]  SERVER → CLIENT  |  RESPONSE  (answers request id=0)
------------------------------------------------------------------
{
  "jsonrpc": "2.0",
  "id": 0,
  "result": {
    "protocolVersion": "2025-11-25",
    "capabilities": {
      "experimental": {},
      "tools": {
        "listChanged": false
      }
    },
    "serverInfo": {
      "name": "ticket-management-server",
      "version": "1.27.0"
    }
  }
}

  [3]  CLIENT → SERVER  |  NOTIFICATION  (no id, one-way, no response expected)
------------------------------------------------------------------
{
  "method": "notifications/initialized",
  "jsonrpc": "2.0"
}

  

## What We Saw

Seven messages for three function calls. That is the complete MCP lifecycle. Let's map them back to the table from the beginning of this section:

| # | Direction | Type | Method / Purpose |
|---|-----------|------|-----------------|
| 1 | CLIENT → SERVER | Request | `initialize`: open connection |
| 2 | SERVER → CLIENT | Response | capabilities + server identity |
| 3 | CLIENT → SERVER | Notification | `notifications/initialized`: ready signal |
| 4 | CLIENT → SERVER | Request | `tools/list`: ask what tools exist |
| 5 | SERVER → CLIENT | Response | four tool definitions with schemas |
| 6 | CLIENT → SERVER | Request | `tools/call`: call `search_tickets` |
| 7 | SERVER → CLIENT | Response | the matching tickets |

A few things worth noticing:

- **Message 3 has no `id`**: it's a notification. The client doesn't wait for a response. It just tells the server it's ready and moves on.
- **Message 5 carries the full schemas.** This is what gets forwarded to the AI model when it's deciding which tool to call.
- **Messages 6 and 7 are the actual work.** Everything before them is infrastructure.

What we ran here used `ClientSession` directly against a single server with in-memory streams. There was no orchestrator and no AI involved. The next section shows what `mcp_client.py` adds on top: it runs this same sequence for all five servers using subprocess stdio transport, collects all 20 tool definitions, and connects an AI model that decides which tools to call.

## Exploring Servers with MCP Inspector

The previous section captured MCP messages manually with a logging wrapper. MCP Inspector gives you the same view through a browser UI. You connect it to any server and get an interface where you can:

- Browse all tools with their full parameter schemas
- Call tools interactively and see responses formatted as JSON
- Watch every JSON-RPC message appear in the History tab in real time

Inspector works against a single server at a time. This makes it a natural continuation of what you've done in this notebook: you explored each server by calling functions directly, you saw the wire messages manually, and now you can do the same exploration visually.

### Local Machine

Open a new terminal tab in the project directory. Install the server dependencies if you haven't already:

```bash
pip3 install -r requirements.txt
```

Then start the Inspector:

```bash
npx @modelcontextprotocol/inspector python3 ticket_server.py
```

Inspector installs itself on the first run (about 30 seconds). It then starts two services:

- A **proxy** on port 6277 that handles communication with the server
- A **web UI** on port 6274 that you open in your browser

Open `http://localhost:6274` in your browser and click **Connect**. To inspect a different server, replace `ticket_server.py` with any of the other four server files.

### Google Colab

Colab doesn't have Node.js installed by default. Open the Colab terminal (the icon in the bottom-left corner) and run:

```bash
pip install -r requirements.txt
apt-get install -y nodejs npm
DANGEROUSLY_OMIT_AUTH=true npx @modelcontextprotocol/inspector python3 ticket_server.py
```

The `DANGEROUSLY_OMIT_AUTH=true` flag skips the session token requirement. Without it, the proxied Colab URL can't authenticate with the Inspector proxy automatically.

Because the Inspector runs on a remote Colab machine, `localhost:6274` won't open in your browser directly. Run the cell below to get a public URL for the web UI. Then click **Connect** in the Inspector UI.

### What to Try First

Once the Inspector is open:

1. The **Tools** tab lists all four ticket server tools. Click `search_tickets` to see the full input schema with every parameter and its description.
2. Fill in `priority: critical` and click **Run Tool**. The result panel shows the response JSON.
3. Open the **History** tab. Every JSON-RPC message from your session appears here: the `initialize` handshake, the `tools/list` response, and each `tools/call` pair. These are the same messages you captured manually in the previous section.

In [15]:
from IPython.display import Image, display
display(Image(url="https://raw.githubusercontent.com/robertbarcik/MCP-tutorial/main/images/inspector_ui.png", width=800))

In [16]:
# Google Colab only: run this cell after starting the Inspector in the Colab terminal.
# It generates a public URL you can open in your browser.
# If you're running locally, open http://localhost:6274 directly instead.

#try:
#    from google.colab.output import eval_js
#    url = eval_js("google.colab.kernel.proxyPort(6274)")
#    print(f"Open MCP Inspector at: {url}")
#except ImportError:
#    print("Not running in Colab. Open http://localhost:6274 in your browser.")

# The Full MCP Protocol in Action

You've just seen what individual MCP messages look like on the wire. The orchestrator in `mcp_client.py` uses those same message types, but it does three things the wire demo didn't:

- It starts **all five servers** as separate subprocesses using stdio transport instead of in-memory streams
- It runs the `initialize` and `tools/list` sequence for each server and collects all 20 tool definitions
- It connects an AI model to the loop so that `tools/call` is triggered by natural language rather than manual code

**Note: `MCPOrchestrator` is custom code we wrote for this tutorial.** The MCP SDK provides the building blocks (`ClientSession`, `stdio_client`, the server-side `Server` class) but does not include anything that coordinates multiple servers or connects them to an AI model. `mcp_client.py` adds that layer: it manages connections to all five servers, converts MCP tool definitions to the format OpenAI expects, and runs the tool-calling loop. In a production system you would build something similar yourself, or use a framework that provides it.

Here's the full flow:

```
1. Orchestrator starts each server as a separate subprocess
   (python3 ticket_server.py, python3 customer_server.py, ...)
                              │
2. Orchestrator sends tools/list to each server
   Servers respond with tool names, descriptions, and parameter schemas
                              │
3. User asks a question in natural language
   "What are the critical tickets for customer CUST-001?"
                              │
4. Orchestrator sends the question + all 20 tool definitions to gpt-5-nano
                              │
5. gpt-5-nano decides which tools to call:
   → search_tickets(customer_id="CUST-001", priority="critical")
                              │
6. Orchestrator routes the call to the correct server (ticket_server)
   and returns the result back to gpt-5-nano
                              │
7. gpt-5-nano may call more tools or formulate a final answer
```

The communication between the orchestrator and each server happens over **stdio** (stdin/stdout). This means the servers run as independent processes and could even be on different machines.

## Running the Interactive Client

The repository includes `interactive_client.py`, a command-line script that runs the full MCP system. It starts all 5 servers, connects them through the orchestrator, and lets you chat with gpt-5-nano using natural language.

This script needs to run from a **terminal**, not from a notebook cell. Here's how to do it in each environment:

**Google Colab:** Click the Terminal icon in the bottom-left corner to open a terminal on the right side. Then run:

```bash
cd /content
pip install -q mcp==1.27.0 nest-asyncio==1.6.0 openai==2.30.0 2>/dev/null
export OPENAI_API_KEY="sk-your-key-here"
python interactive_client.py
```

**Local machine:** Open a terminal in the project directory and run:

```bash
export OPENAI_API_KEY="sk-your-key-here"
python interactive_client.py
```

The `export` sets your API key only for the current terminal session - it disappears when you close the terminal or end your Colab runtime. The script will also prompt you for the key if it's not set.

You can then ask questions like:
- "What are all the critical priority tickets?"
- "Show me customer CUST-001's information and SLA terms"
- "Which customers have both open tickets and overdue invoices?"

gpt-5-nano will automatically discover the 20 available tools, decide which ones to call, and chain multiple calls together to answer complex questions.

# Key Takeaways

Here are the main things to remember from this notebook:

1. **MCP servers are just Python functions underneath** - you can import and call them directly without any protocol infrastructure, which makes testing and debugging straightforward
2. **Error responses are built for AI models, not humans** - they include `suggested_actions`, `follow_up_tools`, and `retryable` flags so the model knows exactly what to try next when something fails
3. **Each server owns its domain** - tickets, customers, billing, knowledge base, and assets are separated into independent servers, but the data is interconnected across them
4. **The MCP protocol adds orchestration on top** - in production, the orchestrator starts servers as subprocesses, discovers their tools automatically, and lets the AI model decide which tools to call based on the user's question

In the next notebook, we'll explore advanced MCP features: **Resources**, **Prompts**, and **Sampling**.